# Random Forest Classification

🎯 **Where this lesson takes you.** In Lesson 2 you built honest *baselines* for an imbalanced bankruptcy problem — a `DummyClassifier` and a plain logistic regression — and learned to read precision, recall, F1, ROC–AUC, and PR–AUC instead of trusting accuracy. Those linear baselines are the floor. This lesson is about climbing above it with a model that captures **nonlinear interactions between financial ratios**: the **Random Forest**, an ensemble of decision trees.

We will train forests, fight class imbalance two different ways (`class_weight="balanced"` and oversampling), check honestly for overfitting with cross-validation, tune a few hyperparameters with `GridSearchCV`, and finally open the black box a little by reading **feature importances**.

## What you'll be able to do

By the end of this notebook you will be able to:

-   Explain Random Forests as an **ensemble** of decision trees built
    with **bagging** (bootstrap aggregation).
-   Train a `RandomForestClassifier` on financial features.
-   Handle class imbalance using a simple **oversampling** strategy.
-   Evaluate performance using metrics that matter for **imbalanced**
    problems: precision, recall, F1, ROC–AUC, and confusion matrices.
-   Compare Random Forest performance to a logistic regression baseline.
-   Interpret **feature importance** scores (Gini importance) and
    discuss limitations.
-   Recognize overfitting risks, model interpretability trade-offs, and
    when Random Forests are effective.

> 💡 **Tip — decoding the features.** The columns are named `feat_1 … feat_64`, which tells you nothing about what they *mean*. If you want the human-readable definition of each `feat_*` column, open `data-dictionary.ipynb`. You don't need it to train a model, but it's invaluable when you later try to interpret which features the forest leaned on.

# 1. Conceptual Foundation

Before we touch the data, let's build the mental model in layers: first *why* we're moving past linear baselines, then the single building block (a decision tree), then the two ideas that turn one tree into a forest (bagging + random feature selection), and finally the practical concerns — imbalance, importance, overfitting, and tuning.

## Why move beyond linear baselines?

In Lesson 2, you trained baseline models such as logistic regression.
Those are useful starting points, but logistic regression is a
**linear** model. Financial ratios can interact in nonlinear ways, so we
now explore a model that can capture nonlinear patterns more naturally:
**Random Forests**.

💡 **Why this matters.** A linear model draws one straight decision boundary through the feature space. But "a firm is risky when leverage is high *and* liquidity is low *and* margins are shrinking" is an **interaction** — the danger only appears when several conditions coincide. Linear models can't express that without you hand-engineering the cross-terms. Tree-based models discover such interactions on their own.

## Decision trees (the building block)

A decision tree repeatedly splits the feature space into regions using
threshold-based rules such as:

-   Is `feat_12 <= 0.2`?
-   If yes, is `feat_7 <= 0.05`?

Trees are intuitive, but a single tree often overfits the training data.

🧠 **The intuition.** A tree is just a flowchart of yes/no questions. Each question carves the data into two cleaner groups; you keep asking until each leaf is (mostly) one class. That flexibility is also the weakness: a tree grown deep enough can memorize the training set — drawing a tiny box around every single bankruptcy — and then fail on new firms. The fix isn't a better single tree; it's *many* trees.

## Bagging (bootstrap aggregation)

Bagging reduces variance by averaging many noisy models:

1.  Sample the training data **with replacement** (bootstrap sample).
2.  Train one model per sample.
3.  Aggregate predictions (majority vote for classification).

🧠 **Why averaging helps.** A single deep tree is *low bias, high variance* — it gets the training data right but swings wildly with small changes in the data. Each bootstrapped tree overfits in its *own* idiosyncratic way; when you average their votes, the random errors cancel out while the real signal reinforces. This is the same logic as asking a crowd to guess and taking the average.

## Random Forests = bagging + random feature selection

Random Forests add extra randomness:

-   At each split, each tree considers only a random subset of features.

This decorrelates trees and often improves generalization.

A Random Forest is precisely those two ingredients stacked together:

| Ingredient | What it randomizes | What it buys you |
|---|---|---|
| **Bagging** | the *rows* each tree sees (bootstrap sample) | lowers variance by averaging |
| **Random feature subsets** | the *columns* each split may use | **decorrelates** the trees so averaging works even harder |

➡️ **Verdict.** Without the second ingredient, every tree would keep splitting on the same one or two dominant features and the trees would look nearly identical — averaging near-identical trees buys little. Forcing each split to choose from a random subset of features makes the trees genuinely different, and *different* errors are what cancel when you vote.

## Imbalance strategies (why oversampling exists)

In bankruptcy prediction, positives are rare. Many models become
conservative and predict mostly zeros.

A simple strategy to help the model see enough bankrupt examples is
**RandomOverSampler**, which duplicates minority-class examples in the
**training set** until classes are balanced.

Important:

-   Oversampling must be applied **only on training data** to avoid
    leakage.
-   We will keep the test set untouched and evaluate fairly.

> ⚠️ **Leakage trap — oversample *after* the split, never before.** If you duplicate minority rows and *then* split, copies of the same firm can land in both train and test. The model effectively sees test answers during training, and your metrics turn into optimistic fiction. The discipline is iron-clad: split first, oversample the **training set only**, evaluate on the **untouched** test set. We'll do exactly that below.

## Feature importance (Gini importance)

Random Forests can estimate which features were most useful for
splitting (based on impurity reduction). This can help exploration, but
it has limitations:

-   correlated features can share importance,
-   importance does not imply causality.

> 🔍 **Read importances as a lead, not a verdict.** Gini importance tells you which features the forest *used* to reduce impurity — handy for exploration. But two highly correlated ratios will split a single "true" importance between them (so both look half as important as the real driver), and a high score never means a feature *causes* bankruptcy. Treat the ranking as a starting point for investigation, not as proof.

## Overfitting checks

We will watch for overfitting by:

-   comparing train vs test performance,
-   using cross-validation (`cross_val_score`) for a more stable
    estimate.

🧠 **The tell.** Overfitting shows up as a *gap*: near-perfect scores on the data the model trained on, noticeably worse scores on data it has never seen. A single train/test split can be lucky or unlucky, so we'll also use **cross-validation** — rotating which slice serves as the held-out fold — to get a steadier read.

## Hyperparameter tuning (simple GridSearchCV)

We will run a small grid search to learn the workflow. The goal is not
to chase a perfect score, but to practice a reliable tuning and
evaluation process.

➡️ **From theory to practice.** That's the whole conceptual toolkit: trees → bagging → random features → imbalance handling → honest evaluation → tuning. Time to load the Poland data and put each idea to work, in order.

------------------------------------------------------------------------

🎥 **Walkthrough video.** Before the hands-on work, watch the short walkthrough for this lesson. Run the cell below to load it.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1170266282", h="3298dbabb7", width=700, height=450)


# Applied Exercises

## 2. Setup

We start by importing everything the lesson needs in one place: pandas for data, matplotlib for the confusion-matrix and importance plots, the imbalanced-learn oversampler, and the scikit-learn estimators, metrics, and model-selection helpers. Recall from Lesson 1 that `wrangle()` lives in `data.py`, so we import it the same way here.

**Code 5.3.2.1**:

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline

from data import wrangle

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 120)

## 3. Load the Poland dataset (full file with target)

### Problem

We need a labeled dataset that includes the bankruptcy target so we can
train and evaluate classification models.

### Approach

Load the **full** Poland file (the one that includes `bankrupt`) and
store the result in `poland_df`. We will reuse `poland_df` throughout
the notebook.

You should reuse the `wrangle()` function from Notebook 1, which is
already defined in `data.py`. If you need to confirm which dataset
contains the target column, see `data-dictionary.ipynb`.

Key points:

-   [`Path`](https://docs.python.org/3/library/pathlib.html#pathlib.Path)
-   [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

**Code 5.3.3.1**:

In [ ]:
poland_path = Path("data/poland-bankruptcy-data-2009.json.gz")
poland_df = wrangle(poland_path)

poland_df.shape

📊 **Reading the shape.** The result is a `(rows, columns)` tuple: one row per Polish firm, and 65 columns — the 64 `feat_*` ratios plus the `bankrupt` target. That's the labeled table every model below will train on.

### Checkpoint

✅ This `assert` is a guardrail: it fails loudly if `bankrupt` is missing, which would mean you loaded the *unlabeled* file by mistake. Passing silently means the target column is present and you're safe to continue.

**Code 5.3.3.2**:

In [ ]:
assert "bankrupt" in poland_df.columns, (
    "Expected 'bankrupt' column. Make sure you loaded the full Poland dataset."
)

## 4. Create `X` (features) and `y` (target)

### Problem

Scikit-learn expects features and target to be separated so we can train
models and evaluate predictions.

### Approach

Identify which columns will be used as model inputs by creating
`feature_cols` as a list of all column names that start with `"feat_"`.
Use this list to build `X`, the feature matrix containing only those
feature columns. Then create `y`, the target vector, by selecting the
`bankrupt` column and converting it to integers so it matches the binary
format expected by scikit-learn.

After creating `X` and `y`, display `X.shape` and `y.shape` to confirm
that the dimensions look correct. Finally, run the checkpoint to verify
that `X` contains the expected number of features and that `y` contains
only binary values `{0, 1}`.

Key points:

-   [`str.startswith`](https://docs.python.org/3/library/stdtypes.html#str.startswith)
-   [`pandas.Series.astype`](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html)
-   Feature/target convention in scikit-learn:
    [`fit(X, y)`](https://scikit-learn.org/stable/glossary.html#term-fit)

**Code 5.3.4.1**:

In [ ]:
feature_cols = [c for c in poland_df.columns if c.startswith("feat_")]

X = poland_df[feature_cols]
y = poland_df["bankrupt"].astype(int)

X.shape, y.shape

📊 **Reading the shapes.** `X` should have 64 columns (one per feature) and the same number of rows as `y`. Keeping `feature_cols` as an explicit list pays off later: we reuse it to label the feature-importance chart so each bar maps back to a real column name.

### Checkpoint

✅ Two guardrails at once: the feature matrix must have exactly 64 columns, and the target must be strictly binary `{0, 1}`. If either fails, something upstream went wrong and every model below would be training on the wrong shape.

**Code 5.3.4.2**:

In [ ]:
assert X.shape[1] == 64, "Expected 64 Poland features (feat_1..feat_64)."
assert set(y.unique()).issubset({0, 1}), "Target must be binary {0, 1}."

## 5. Train/test split (stratified)

### Problem

We need an unbiased test set for evaluation, but we also need to
preserve the rare bankruptcy rate in both splits.

### Approach

Split the dataset into training and test sets in a way that preserves
the rare bankruptcy rate in both splits. Use `X` and `y` as inputs and
create `X_train`, `X_test`, `y_train`, and `y_test` with a fixed
`random_state` for reproducibility and a test size of 25%. Make sure the
split is stratified so the class proportions remain similar across the
two subsets.

After the split, inspect `X_train.shape` and `X_test.shape` to confirm
the sizes look reasonable. Then compute the mean of `y_train` and
`y_test` and store them in `train_rate` and `test_rate`. Finally, run
the checkpoint to verify that the difference between `train_rate` and
`test_rate` is small, confirming that the class rate was preserved.

Key points:

-   [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
-   Stratification:
    [`stratify`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

> ⚠️ **Why `stratify=y` is non-negotiable here.** With bankruptcies this rare, an ordinary random split can hand one side far fewer positives than the other — or, in a small split, almost none. Stratifying forces the *same* bankruptcy rate into both train and test, so your test metrics actually reflect the problem you're modeling.

**Code 5.3.5.1**:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape

### Checkpoint: class rate preserved

✅ This is the payoff of `stratify=y`: the bankruptcy rate in train and test should differ by less than one percentage point. If it didn't, your test set wouldn't be a fair stand-in for reality.

**Code 5.3.5.2**:

In [ ]:
train_rate = y_train.mean()
test_rate = y_test.mean()

train_rate, test_rate

assert abs(train_rate - test_rate) < 0.01, (
    "Stratified split should preserve the bankruptcy rate."
)

## 6. Baseline model (Logistic Regression) for comparison

### Problem

Before using a nonlinear model, we want a simple baseline to compare
against.

### Approach

We use a **simple** Logistic Regression baseline as a reference point
and do not apply imbalance-specific tuning (e.g.,
`class_weight="balanced"` or threshold optimization). This keeps the
baseline easy to interpret and makes the Random Forest comparison
straightforward.

In this section, you will build a simple baseline model that you can use
as a reference for later nonlinear models. Start by creating a pipeline
named `baseline_lr` that first imputes missing values using the median
and then fits a logistic regression model with suitable training
settings. Fit this pipeline on the training split (`X_train`,
`y_train`), then generate test-set class predictions in `y_pred_lr` and
positive-class probabilities in `y_proba_lr`.

Next, evaluate the baseline on the test set by computing precision,
recall, F1, and ROC–AUC (use `zero_division=0` for the classification
metrics and use `y_proba_lr` for ROC–AUC). Store these results in a
dictionary called `baseline_metrics`. Finally, visualize the baseline
performance by plotting a confusion matrix using `y_test` and
`y_pred_lr`.

Key points:

-   [`make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
-   [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)
-   [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
-   [predict](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict)
-   [predict_proba](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict_proba)
-   [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
-   [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
-   [f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
-   [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)
-   [ConfusionMatrixDisplay.from_predictions](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)

🧱 **Why bother with a baseline at all?** A baseline is the "could a simple model already do this?" bar. Every fancier model has to *beat* it to justify its complexity. A `median` imputer is the minimum needed because logistic regression can't accept `NaN`s; we deliberately skip imbalance tricks here so the comparison with the forest stays clean.

**Code Task 5.3.6.1**:

In [ ]:
baseline_lr = make_pipeline(
    # instantiate the simple imputer class with the appropriate arguments
    SimpleImputer(strategy='median'),
    # instantiate the logistic regression class with the appropriate arguments
    LogisticRegression(max_iter=1000, solver='lbfgs'),
)

# fit baseline_lr using X_train and y_train
baseline_lr.fit(X_train, y_train)

# predict the target from X_test using the aseline_lr model
y_pred_lr = baseline_lr.predict(X_test)
# gets the predicted probability of the positive class (1) from the baseline_lr model.
y_proba_lr = baseline_lr.predict_proba(X_test)[:, 1]


### Baseline metrics (test set)

📊 We summarize the baseline in the four numbers that matter for an imbalanced problem. Watch **recall** especially: a plain logistic regression on rare positives often plays it safe and predicts "not bankrupt" too often, which shows up as low recall even when accuracy looks fine.

**Code 5.3.6.2**:

In [ ]:
baseline_metrics = {
    "precision": precision_score(y_test, y_pred_lr, zero_division=0),
    "recall": recall_score(y_test, y_pred_lr, zero_division=0),
    "f1": f1_score(y_test, y_pred_lr, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba_lr),
}
baseline_metrics

### Confusion matrix (baseline)

📊 **What to look for.** The off-diagonal bottom-left cell is **false negatives** — bankrupt firms the baseline labeled "safe." On this problem those are the costly mistakes (you cleared a firm that later collapsed). Note how many there are; it's the number every later model will try to shrink.

**Code 5.3.6.3**:

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_lr,
    values_format="d",
)
plt.title("Logistic Regression — Confusion Matrix (Test)")
plt.show()

## 7. Random Forest

### Problem

We want a nonlinear model that can capture interactions and complex
patterns without manually engineering them.

### Approach

Build a simple Random Forest pipeline that can handle missing values and
class imbalance out of the box. First, create `rf_simple` as a pipeline
with a median imputer followed by a `RandomForestClassifier` configured
with `class_weight="balanced"`, `n_estimators=300`, `random_state=42`,
and `n_jobs=-1`. Fit `rf_simple` on `X_train` and `y_train`. Then
generate test-set predictions in `y_pred_rf` and store the
positive-class probabilities in `y_proba_rf`. Evaluate the model on the
test set by computing precision, recall, F1, and ROC–AUC (use
probabilities for ROC–AUC) and store them in `rf_metrics`. Finally,
visualize the confusion matrix and compare `rf_metrics` against the
logistic regression baseline in a table called `compare`.

Key points:

-   [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
-   `class_weight="balanced"`: [class
    weights](https://scikit-learn.org/stable/glossary.html#term-class_weight)
-   Trees do not require scaling: [tree-based
    models](https://scikit-learn.org/stable/modules/tree.html)
-   [`make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
-   [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)
-   [`RandomForestClassifier.predict`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier.predict)
-   [`RandomForestClassifier.predict_proba`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier.predict_proba)
-   [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
-   [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
-   [f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
-   [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)
-   [ConfusionMatrixDisplay.from_predictions](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)

🔧 **Two knobs worth naming.** `class_weight="balanced"` tells the forest to *penalize* mistakes on the rare class more heavily — the first of our two imbalance strategies (the second, oversampling, comes in §8). And unlike logistic regression, trees split on thresholds, so they're scale-invariant: no `MinMaxScaler` needed, just the median imputer to handle `NaN`s.

**Code Task 5.3.7.1**:

In [ ]:
# build a preprocessing + model pipeline:
# - impute missing values using the median
# - train a random forest with balanced class weights for imbalanced data
rf_simple = make_pipeline(
    SimpleImputer(strategy='median'),
    RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced',
    ),
)
# fit the pipeline on the training split
rf_simple.fit(X_train, y_train)

# predict class labels on the test split
y_pred_rf = rf_simple.predict(X_test)
# predict probabilities on the test split and select class 1 probabilities
y_proba_rf = rf_simple.predict_proba(X_test)[:, 1]


### Evaluate (test set)

📊 Same four metrics as the baseline, so the comparison is apples-to-apples. With `class_weight="balanced"`, expect recall to move relative to the baseline — that's the whole point of weighting the rare class more heavily.

**Code 5.3.7.2**:

In [ ]:
# compute core metrics on the test set using class predictions
# note: use `zero_division=0` to avoid warnings when no positives are predicted
rf_metrics = {
    "precision": precision_score(y_test, y_pred_rf, zero_division=0),
    "recall": recall_score(y_test, y_pred_rf, zero_division=0),
    "f1": f1_score(y_test, y_pred_rf, zero_division=0),
    # ROC–AUC must use probabilities, not hard class labels
    "roc_auc": roc_auc_score(y_test, y_proba_rf),
}

rf_metrics

### Confusion matrix (Random Forest)

📊 Compare this grid cell-by-cell against the baseline's. The question isn't "did total errors drop?" but "did the **false negatives** (missed bankruptcies) shrink, and at what cost in false positives?"

**Code 5.3.7.3**:

In [ ]:
# plot the confusion matrix to see TP/FP/TN/FN counts at the default threshold
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_rf,
    values_format="d",
)
# add a readable title and render the plot
plt.title("Random Forest — Confusion Matrix (Test)")
plt.show()

### Compare to baseline

📊 Putting both models in one table makes the trade-off explicit. A forest often trades a little precision for a meaningful gain in recall (or vice versa) — read the row difference, not any single cell.

**Code 5.3.7.4**:

In [ ]:
# build a compact comparison table with the baseline and random forest metrics
compare = pd.DataFrame(
    [
        {"model": "LogReg baseline", **baseline_metrics},
        {"model": "Random Forest", **rf_metrics},
    ]
).set_index("model")

compare

### Train vs test metrics (Random Forest)

🔍 **The overfitting check.** Here we score the *same* forest on the data it trained on and on the held-out test set. Random Forests with deep trees can post near-perfect **train** numbers; what matters is how far the **test** numbers fall short. A large gap is the signature of overfitting — and motivates the cross-validation and tuning that follow.

**Code 5.3.7.5**:

In [ ]:
# compute the same metrics on train vs test to check for overfitting
# note: ROC–AUC must use probabilities

y_pred_rf_train = rf_simple.predict(X_train)
y_proba_rf_train = rf_simple.predict_proba(X_train)[:, 1]

rf_train_metrics = {
    "precision": precision_score(y_train, y_pred_rf_train, zero_division=0),
    "recall": recall_score(y_train, y_pred_rf_train, zero_division=0),
    "f1": f1_score(y_train, y_pred_rf_train, zero_division=0),
    "roc_auc": roc_auc_score(y_train, y_proba_rf_train),
}

rf_train_metrics

**Code 5.3.7.6**:

In [ ]:
# compare train vs test side-by-side
rf_train_test = pd.DataFrame(
    [rf_train_metrics, rf_metrics],
    index=["train", "test"],
)

rf_train_test

📊 **Reading the gap.** Side by side, the `train` row will likely look much stronger than the `test` row. That distance is your overfitting budget: the forest memorized training quirks it can't reproduce on new firms. We don't "fix" it by trusting the train numbers — we measure performance honestly on test and, next, with cross-validation.

**Code 5.3.7.7**:

In [ ]:
assert set(rf_train_test.columns) == {"precision", "recall", "f1", "roc_auc"}, (
    "Expected the standard metric columns in the train/test comparison table."
)

✅ A structural guardrail: the train/test comparison table must carry exactly the four metric columns we expect. It protects the rows above from a silent typo in a metric key.

## 8. Random oversampling (to address imbalance)

### Problem

With a rare positive class, the model may under-predict bankruptcies.
Oversampling can improve recall by giving the model more positive
examples to learn from.

### Approach

First, create an oversampled version of the **training set only** so the
model sees more positive examples during training without leaking
information from the test set. Use `RandomOverSampler` to generate
`X_train_over` and `y_train_over`, and inspect the new shape and class
counts to confirm the resampling worked. Then run a checkpoint to verify
the oversampled training labels are balanced. Next, train a Random
Forest pipeline on the oversampled training data by imputing missing
values and fitting the classifier. After training, evaluate the model on
the **unchanged test set** by generating predictions in `y_pred_rf_over`
and positive-class probabilities in `y_proba_rf_over`. Compute the main
metrics in `rf_over_metrics`, plot a confusion matrix for the test-set
results, and finally build `compare2` to compare the baseline, the
balanced class-weight forest, and the oversampled forest side by side.

Key points:

-   [`make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
-   [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)
-   [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
-   [`RandomForestClassifier.predict`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier.predict)
-   [`RandomForestClassifier.predict_proba`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier.predict_proba)
-   [`RandomOverSampler`](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.RandomOverSampler.html)
-   [`RandomOverSampler.fit_resample`](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.RandomOverSampler.html#imblearn.over_sampling.RandomOverSampler.fit_resample)
-   [resampling
    guidelines](https://imbalanced-learn.org/stable/over_sampling.html)
-   [`value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
-   [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
-   [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
-   [f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
-   [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)
-   [ConfusionMatrixDisplay.from_predictions](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)

🔄 **Two routes to the same goal.** `class_weight="balanced"` (§7) and `RandomOverSampler` (here) both try to make the rare class matter more. Weighting *re-prices* each mistake; oversampling *duplicates* minority rows until the counts even out. They often land in similar territory — which is exactly why we'll benchmark both against the baseline in `compare2`.

**Code Task 5.3.8.1**:

In [ ]:
# create the oversampler (fixed seed for reproducibility)
ros = RandomOverSampler(random_state=42)
# oversample the training split only (never oversample the test set)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

# inspect the new training shape and class balance after resampling
X_train_over.shape, pd.Series(y_train_over).value_counts()


📊 **Reading the resampled counts.** After `fit_resample`, the two class counts in `value_counts()` should be equal, and the training set is now larger than before — the extra rows are duplicated bankruptcies. Crucially, this happened on `X_train`/`y_train` only; the test set is untouched.

### Checkpoint: balanced classes after resampling

✅ This `assert` confirms the oversampler did its job: the two class counts must now be identical. If they weren't, the "balanced" training set wouldn't actually be balanced.

**Code 5.3.8.2**:

In [ ]:
# compute class counts after oversampling
counts_over = pd.Series(y_train_over).value_counts()
# checkpoint: after random oversampling, both classes should have the same count
assert counts_over.iloc[0] == counts_over.iloc[1], (
    "Oversampled training set should have balanced classes."
)

### Train Random Forest on oversampled data

🔧 Note this forest drops `class_weight="balanced"` — the data is already balanced by duplication, so re-weighting on top would double-count the fix. We still impute with the median, then fit on the **oversampled** training split and predict on the **original** test split.

**Code 5.3.8.3**:

In [ ]:
# build a pipeline:
# - impute missing values using the median (required for tree models in sklearn)
# - fit a random forest on the oversampled training data
rf_over = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),
)
# train the pipeline on the oversampled training split
rf_over.fit(X_train_over, y_train_over)

# predict class labels on the original (unchanged) test split
y_pred_rf_over = rf_over.predict(X_test)
# predict probabilities on the test split and select class 1 probabilities
y_proba_rf_over = rf_over.predict_proba(X_test)[:, 1]

### Evaluate oversampled model (test set)

📊 The four metrics again, computed on the untouched test set. Compare recall here against the `class_weight="balanced"` forest from §7 — the interesting question is whether duplication buys recall that weighting didn't.

**Code 5.3.8.4**:

In [ ]:
# compute core metrics on the unchanged test set
# note: use probabilities for ROC–AUC and `zero_division=0` for stability
rf_over_metrics = {
    "precision": precision_score(y_test, y_pred_rf_over, zero_division=0),
    "recall": recall_score(y_test, y_pred_rf_over, zero_division=0),
    "f1": f1_score(y_test, y_pred_rf_over, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba_rf_over),
}

rf_over_metrics

### Confusion matrix (RF trained with RandomOverSampler — test set)

📊 Once more, watch the false-negative cell. Oversampling is meant to push the model toward catching more positives; the matrix shows whether it did, and how many false alarms it cost.

**Code 5.3.8.5**:

In [ ]:
# visualize TP/FP/TN/FN counts for the oversampled-training model on the test set
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_rf_over,
    values_format="d",
)
# add a readable title and render the plot
plt.title("Random Forest (Oversampled Train) — Confusion Matrix (Test)")
plt.show()

### Compare all models

📊 **The three-way verdict.** `compare2` lines up the baseline, the weighted forest, and the oversampled forest. No single row "wins" on every metric — pick the model whose precision/recall balance matches what the business actually cares about (here, catching bankruptcies, i.e. recall, usually outranks avoiding false alarms).

**Code 5.3.8.6**:

In [ ]:
# build a comparison table across models using the same metric keys
# assumes `baseline_metrics` and `rf_metrics` were computed earlier
compare2 = pd.DataFrame(
    [
        {"model": "LogReg baseline", **baseline_metrics},
        {"model": "RF (class_weight=balanced)", **rf_metrics},
        {"model": "RF (RandomOverSampler)", **rf_over_metrics},
    ]
).set_index("model")

# display the comparison table
compare2

## 9. Cross-validation (more stable estimate)

### Problem

A single train/test split can be noisy. Cross-validation provides a more
stable estimate of model performance.

### Approach

Estimate the model’s performance more reliably by running
cross-validation on the **training set**. Use `rf_simple` as the
estimator and compute ROC–AUC for each fold using 5-fold
cross-validation. Store the per-fold scores in `scores`, then summarize
them by computing the mean and standard deviation. Finally, run a
checkpoint to confirm that you obtained exactly 5 scores (one per fold).

Key points:

-   [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
-   ROC–AUC scoring:
    [`roc_auc`](https://scikit-learn.org/stable/modules/model_evaluation.html#roc-metrics)

🔍 **Why five reads beat one.** A single split gives you one number with no sense of its uncertainty. 5-fold cross-validation rotates which fifth of the training data is held out, giving five ROC–AUC scores. Their **mean** is a steadier estimate; their **standard deviation** tells you how much the model's quality wobbles with the luck of the split.

**Code Task 5.3.9.1**:

In [ ]:
# run 5-fold cross-validation on the training set
# scoring='roc_auc' computes ROC–AUC using predicted probabilities internally
scores = cross_val_score(
    rf_simple,
    X_train,
    y_train,
    cv=5,
    scoring='roc_auc',
)

# inspect per-fold ROC–AUC scores and summarize with mean and standard deviation
scores, scores.mean(), scores.std()


📊 **Reading the folds.** Look at two things: the **mean** (your best single estimate of out-of-sample ROC–AUC) and the **spread** across the five folds. A tight spread means the estimate is trustworthy; a wide one means performance depends heavily on which firms land in the held-out fold.

### Checkpoint

✅ A simple structural check: 5-fold cross-validation must return exactly five scores. If it didn't, the `cv` argument wasn't wired up correctly.

**Code 5.3.9.2**:

In [ ]:
# checkpoint: 5-fold cross-validation must return exactly 5 scores
assert len(scores) == 5, "Expected 5-fold cross-validation scores."

## 10. Simple hyperparameter tuning with GridSearchCV

### Problem

Default hyperparameters are rarely optimal. We want a structured way to
try a few settings and select the best one using cross-validation.

### Approach

Start by defining a reusable pipeline called `rf_pipe` that imputes
missing values with the median and trains a balanced Random Forest.
Next, define `param_grid` to specify the small set of hyperparameters
you want to try for the Random Forest step inside the pipeline. Then run
a grid search called `grid` using ROC–AUC scoring and 3-fold
cross-validation on the training set, and fit it with `X_train` and
`y_train`. After training, inspect `grid.best_params_` and
`grid.best_score_` to see which configuration performed best in
cross-validation. Finally, extract the best tuned model (`best_rf`),
evaluate it on the unchanged test set by producing `y_pred_best` and
`y_proba_best`, compute `best_metrics`, plot the confusion matrix, and
compare the tuned model to the baseline in `compare3`.

Key points:

-   [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
-   [pipeline
    parameters](https://scikit-learn.org/stable/modules/compose.html#using-pipeline-with-grid-search)
-   [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
-   [`make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
-   [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)
-   [`RandomForestClassifier.predict`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier.predict)
-   [`RandomForestClassifier.predict_proba`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html#sklearn.ensemble.RandomForestClassifier.predict_proba)
-   [precision_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html)
-   [recall_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)
-   [f1_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
-   [roc_auc_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)
-   [ConfusionMatrixDisplay.from_predictions](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.from_predictions)

🔧 **The `<step>__<param>` convention.** Inside a pipeline you can't just say `n_estimators`; you prefix it with the step name, e.g. `randomforestclassifier__n_estimators`, so the grid search knows *which* step to tune. The three knobs here — number of trees, max depth, and min samples per leaf — are the classic levers for trading off forest capacity against overfitting.

**Code 5.3.10.1**:

In [ ]:
# define a simple preprocessing + model pipeline:
# - impute missing values using the median
# - train a random forest with balanced class weights for imbalanced data
rf_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
    ),
)

# define the hyperparameters for randomforestclassifier
# note: pipeline parameters use the "<step>__<param>" naming convention
# - n_estimators = [200, 400]
# - max_depth = [None, 8, 16]
# - min_samples_leaf = [1, 5, 10]
param_grid = {
    "randomforestclassifier__n_estimators": [200, 400],
    "randomforestclassifier__max_depth": [None, 8, 16],
    "randomforestclassifier__min_samples_leaf": [1, 5, 10],
}

# configure the grid search:
# - scoring="roc_auc" evaluates models using ROC–AUC
# - cv=3 performs 3-fold cross-validation on the training set
# - n_jobs=-1 parallelizes across CPU cores
grid = GridSearchCV(
    rf_pipe,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
)

# run the grid search (fit) on the training split
grid.fit(X_train, y_train)

# inspect the best hyperparameters and their mean cross-validation ROC–AUC score
grid.best_params_, grid.best_score_

📊 **Reading the search result.** `best_params_` is the winning combination of the three knobs; `best_score_` is its mean cross-validated ROC–AUC. Remember this score is measured on held-out folds *inside* the training set — it's an honest estimate, not a test-set number. The real test comes next.

### Evaluate best model on test set

📊 Now we take the tuned model to the **untouched** test set. The fair question: does `best_metrics` actually beat the default forest, or did tuning just chase noise in the cross-validation folds?

**Code 5.3.10.2**:

In [ ]:
# extract the best tuned pipeline found during cross-validation
# note: attribute best_estimator_ from `grid`
best_rf = grid.best_estimator_

# generate predictions on the unchanged test split
y_pred_best = best_rf.predict(X_test)
# generate probabilities on the test split and select class 1 probabilities
y_proba_best = best_rf.predict_proba(X_test)[:, 1]

# compute test-set metrics for the tuned model
# note: ROC–AUC must use probabilities, not hard class labels
best_metrics = {
    "precision": precision_score(y_test, y_pred_best, zero_division=0),
    "recall": recall_score(y_test, y_pred_best, zero_division=0),
    "f1": f1_score(y_test, y_pred_best, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba_best),
}

# display the metric summary
best_metrics

### Confusion matrix (tuned Random Forest)

📊 The tuned model's error breakdown. Compare the false-negative cell against the earlier forests — tuning for ROC–AUC doesn't automatically minimize missed bankruptcies, so check whether the cell you care about actually improved.

**Code 5.3.10.3**:

In [ ]:
# visualize TP/FP/TN/FN counts for the tuned model on the test set
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_best,
    values_format="d",
)

# add a readable title and render the plot
plt.title("Best Random Forest (GridSearchCV) — Confusion Matrix (Test)")
plt.show()

### Compare baseline vs tuned Random Forest

📊 `compare3` closes the modeling arc: the simplest model (logistic baseline) against the most carefully tuned forest. This is the headline comparison — how much did all the nonlinearity, imbalance handling, and tuning actually buy over the floor we started from?

**Code 5.3.10.4**:

In [ ]:
# build a compact comparison table between the baseline and tuned random forest
# assumes `baseline_metrics` was computed earlier
compare3 = pd.DataFrame(
    [
        {"model": "LogReg baseline", **baseline_metrics},
        {"model": "RF (tuned)", **best_metrics},
    ]
).set_index("model")

# display the comparison table
compare3

## 11. Feature importance (Gini importance)

### Problem

We want a simple way to understand which features the Random Forest
relied on most (as measured by impurity reduction).

### Approach

Start from the tuned pipeline stored in `best_rf` and extract the fitted
Random Forest step into `rf_model`. Then build a `pandas.Series` called
`importances` using the model’s importance values, and index it by
`feature_cols` so each importance is aligned with its feature name. Sort
`importances` in descending order and inspect the top values to see
which features contribute the most. Next, choose a number of top
features (`top_n`) and plot them as a horizontal bar chart to visualize
the relative importance. Finally, run the checkpoint to ensure you have
one importance value per feature and that all importance values are
within the expected range.

Key points:

-   `feature_importances_` attribute: [scikit-learn
    docs](https://scikit-learn.org/stable/modules/ensemble.html#feature-importance)
-   [`pandas.Series`](https://pandas.pydata.org/docs/reference/api/pandas.Series.html)
-   Bar plots in pandas:
    [`Series.plot`](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.html)

🔍 **Indexing by `feature_cols` is the trick.** `feature_importances_` comes back as a bare array in column order. Pairing it with `feature_cols` as the index turns anonymous positions into named features, so the sorted Series and the chart below are actually interpretable.

**Code Task 5.3.11.1**:

In [ ]:
# extract the fitted random forest model from the tuned pipeline
# check the steps in `best_rf` via `best_rf.named_steps`
# and select the step for random forest classifier
rf_model = best_rf.named_steps['randomforestclassifier']

# build a Series of feature importances indexed by the feature column names
# sort descending so the most important features appear first
importances = pd.Series(
    # use the `feature_importances_` attribute from `rf_model`
    rf_model.feature_importances_,
    index=feature_cols,
).sort_values(ascending=False)

# inspect the top 10 most important features
importances.head(10)

📊 **Reading the top features.** These are the ratios the forest leaned on most to reduce impurity. Resist over-reading them: per the caveat above, correlated ratios split their importance, so a feature ranked #3 might be part of the same underlying signal as #1.

### Plot top 20

📊 The horizontal bar chart makes the *shape* of importance visible — does a handful of features dominate, or is importance spread thinly across many? A steep drop-off suggests a few ratios carry most of the predictive weight.

**Code 5.3.11.2**:

In [ ]:
# choose the top 20 features to visualize
top_n = 20

# plot the top importances as a horizontal bar chart (sorted for readability)
ax = importances.head(top_n).sort_values().plot(kind="barh")

# label the plot clearly
ax.set_title(f"Top {top_n} feature importances (Random Forest)")
ax.set_xlabel("Gini importance")
plt.show()

### Checkpoint

✅ Two final guardrails: there must be exactly one importance value per feature column, and (since Gini importances are normalized) none may exceed 1. Together they confirm the Series is well-formed before you trust the chart.

**Code 5.3.11.3**:

In [ ]:
# checkpoint: there must be one importance value per feature column
assert importances.shape[0] == len(feature_cols), (
    "Importances should exist for every feature column."
)

# checkpoint: importances are normalized and should not exceed 1
assert importances.max() <= 1.0 + 1e-12, "Importances should be <= 1."

## 12. Apply the same pipeline to Taiwan (Optional/Ungraded)

In this section, you can apply what you learned in the previous sections
to analyze the Taiwan dataset. This section is optional and ungraded,
but it is highly recommended to reinforce what you have learned so far.

💡 **Why repeat on a second dataset?** The Taiwan bankruptcy data has a different feature set and a different imbalance ratio. Re-running the same split → baseline → forest → evaluate workflow there is the best way to confirm you've internalized the *process*, not just memorized the Poland numbers.

**Code 5.3.12.1**:

In [ ]:
# your code here

------------------------------------------------------------------------

# Wrap-up

In this notebook you:

-   Learned Random Forests as a **bagging-based ensemble** method.
-   Trained Random Forest classifiers on financial features.
-   Evaluated models using imbalanced metrics and confusion matrices.
-   Compared Random Forest performance to a logistic regression
    baseline.
-   Used **RandomOverSampler** to address imbalance (training only).
-   Used cross-validation and a simple **GridSearchCV** workflow to tune
    the model.
-   Interpreted Gini feature importances while understanding their
    limitations.

➡️ **Where this goes next.** You now have a strong nonlinear model and an honest way to evaluate it. In Lesson 4 you'll meet the other great ensemble family — **boosting** — and put Random Forests, gradient boosting, and the linear baseline head-to-head, then wrap the winner into a deployable end-to-end workflow. Next, you will compare Random Forests with **boosting** methods and build a more complete end-to-end modeling workflow.